In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
SAMPLE = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
TRAIN = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
TEST = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [5]:
len(TRAIN)

2000

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class MCQDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_length

        self.label_map = {
            "A": 0,
            "B": 1,
            "C": 2,
            "D": 3,
            "E": 4
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        text = (
            f"Question: {row['prompt']} "
            f"A. {row['A']} "
            f"B. {row['B']} "
            f"C. {row['C']} "
            f"D. {row['D']} "
            f"E. {row['E']}"
        )

        encoding = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        label = self.label_map[row["answer"]]

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(TRAIN, test_size=0.2, random_state=42, stratify=TRAIN["answer"])

In [9]:
train_dataset = MCQDataset(data=train_df, tokenizer=tokenizer, max_length=256)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [10]:
len(train_dataset.tokenizer)

30522

In [11]:
import torch.nn as nn 

class BiLSTM(nn.Module):
    def __init__(self, vocab_size=len(train_dataset.tokenizer), embed_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)

        self.fc = nn.Linear(hidden_dim*2, 5)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        output, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(hidden)

In [29]:
from torch.optim import AdamW

learning_rate = 1e-3
epochs = 20
model = BiLSTM()
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

In [30]:
wandb.init(
    project="Smart_MCQ_solver",
    name="bi_llstm-run2",
    config={
        "epochs": epochs,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "batch_size": train_dataloader.batch_size,
        "model": "bi-LSTM"
    }
)

In [31]:
model = model.to(device)
model.train()

global_step = 0
for epoch in range(epochs):
    total_loss = 0

    for batch in train_dataloader:
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        output = model(input_ids)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        wandb.log({
            "train/batch_loss": loss.item(),
            "learning_rate": learning_rate,
            "global_step": global_step
        })
        global_step += 1
    avg_loss = total_loss / len(train_dataloader)    
    print(f"epoch : {epoch + 1}, Loss : {total_loss: .4f}")
    wandb.log({"train/epoch_loss": avg_loss, "epoch": epoch + 1})

epoch : 1, Loss :  80.0872
epoch : 2, Loss :  74.2439
epoch : 3, Loss :  60.6378
epoch : 4, Loss :  56.1600
epoch : 5, Loss :  40.1922
epoch : 6, Loss :  24.4637
epoch : 7, Loss :  9.3445
epoch : 8, Loss :  5.1286
epoch : 9, Loss :  1.8560
epoch : 10, Loss :  0.8517
epoch : 11, Loss :  0.3542
epoch : 12, Loss :  0.2347
epoch : 13, Loss :  0.1824
epoch : 14, Loss :  0.1491
epoch : 15, Loss :  0.1235
epoch : 16, Loss :  0.1055
epoch : 17, Loss :  0.0917
epoch : 18, Loss :  0.0803
epoch : 19, Loss :  0.0702
epoch : 20, Loss :  0.0610


In [32]:
test_dataset = MCQDataset(data=test_df, tokenizer=tokenizer, max_length=256)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [33]:
model.eval()

correct = 0
total = 0
map3 = 0.0
val_loss = 0

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        top3 = outputs.topk(3, dim=1).indices

        for i in range(labels.size(0)):
            true_label = labels[i].item()
            predictions = top3[i].tolist()

            if true_label == predictions[0]:
                map3 += 1.0
            elif true_label == predictions[1]:
                map3 += 0.5
            elif true_label == predictions[2]:
                map3 += 1.0 / 3.0

accuracy = correct / total
map3 /= total


print(f"Validation Accuracy : {accuracy:.4f}")
print(f"Validation MAP@3    : {map3:.4f}")
wandb.log({
    "val/accuracy": accuracy,
    "val/MAP@3": map3
})


Validation Accuracy : 0.9875
Validation MAP@3    : 0.9929


In [34]:
wandb.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
global_step,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/batch_loss,████▆▆▅▆▆▅▅▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_loss,█▇▆▆▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/MAP@3,▁
val/accuracy,▁
epoch,20
global_step,999
learning_rate,0.001
train/batch_loss,0.00112


In [35]:
from sklearn.metrics import confusion_matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        outputs = model(batch["input_ids"].to(device))
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(batch["labels"].numpy())

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[72  0  1  0  1]
 [ 0 98  0  0  0]
 [ 0  0 91  1  0]
 [ 0  0  0 71  0]
 [ 1  1  0  0 63]]


In [36]:
from torch.utils.data import Dataset

class BiLSTMTestDataset(Dataset):

    def __init__(self, data, tokenizer, max_length):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        text = (
            f"Question: {row['prompt']} "
            f"A. {row['A']} "
            f"B. {row['B']} "
            f"C. {row['C']} "
            f"D. {row['D']} "
            f"E. {row['E']}"
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [37]:
test_dataset = BiLSTMTestDataset(TEST, tokenizer, max_length=256)

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [38]:
batch = next(iter(test_dataloader))
print(batch["input_ids"].shape)

torch.Size([32, 256])


In [39]:
model.eval()

label_map = ["A", "B", "C", "D", "E"]
predictions = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)

        logits = model(input_ids)

        top3 = torch.topk(logits, k=3, dim=1).indices.cpu().numpy()

        for pred in top3:
            predictions.append(" ".join(label_map[i] for i in pred))

In [40]:
submission = pd.DataFrame({
    "id": TEST["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,prediction
0,1,A B E
1,2,B A E
2,3,B A E
3,4,E A C
4,5,C A D
